# SQL in Python - Connecting to and retrieving data from PostgreSQL

Previously, you have learned how to connect to a SQL database by using a SQL client such as DBeaver. Apart from connecting to databases, DBeaver also allows you to run SQL queries against the database, create new tables and populate them with data as well as retrieving the data.

Python also allows executing SQL queries and getting the result into a Python object, for example a Pandas data frame. Instead of exporting a .csv file from DBeaver you can directly get the data you need into Python and continue your work. In addition we can reduce the steps by connecting to the database from Python directly, eliminating the need for a separate SQL client.

After you have the data in Python in the required shape you can export the data into a .csv file. This file is for your own reference, please avoid sending .csv files around - database is the point of reference when it comes to data. 

Having a copy of a .csv file (or another format) can speed up your analysis work. Imagine that the query takes 25 minutes to run. If you made some mistakes in your Python code you might need to go back to the original dataset. Instead of having to rerun the SQL query and having to wait you can read in the .csv file you have previously saved on your hard disk into Python and continue with your analysis work. 

**In this notebook you will see 2 ways to connect to SQL-Databases and export the data to a CSV file**


## Creating a connection to a PostgreSQL database with Python

There are 2 python packages that are the "go-to" when it comes to connecting to SQL-Databases: `psycopg2` and `sqlalchemy` 

### Connecting via psycopg2

In [16]:
%pip install psycopg2-binary


Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import psycopg2


In order to create a connection to our PostgreSQL database we need the following information:

- host = the address of the machine the database is hosted on
- port = the virtual gate number through which communication will be allowed
- database = the name of the database
- user = the name of the user
- password = the password of the user

Because we don't want that the database information is published on GitHub we put it into a `.env` file which is added into the `.gitignore`. 
In these kind of files you can store information that is not supposed to be published.
With the `dotenv` package you can read the `.env` files and get the variables.


In [18]:
pip install python-dotenv

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
from dotenv import load_dotenv

load_dotenv()

DATABASE = os.getenv("DATABASE")
USER_DB = os.getenv("USER_DB")
PASSWORD = os.getenv("PASSWORD")
HOST = os.getenv("HOST")
PORT = os.getenv("PORT")

print(DATABASE)

postgres


In [4]:
DATABASE = os.getenv("DATABASE")
USER_DB = os.getenv("USER_DB")
PASSWORD = os.getenv("PASSWORD")
HOST = os.getenv("HOST")
PORT = os.getenv("PORT")

The function from the psycopg2 package to create a connection is called `connect()`.
`connect()` expects the parameters listed above as input in order to connect to the database.

In [5]:
# Create connection object conn
conn = psycopg2.connect(
    database=DATABASE, user=USER_DB, password=PASSWORD, host=HOST, port=PORT
)

### Retrieving data from the database with psycopg2

Before we can use our connection to get data, we have to create a cursor. A cursor allows Python code to execute PostgreSQL commands in a database session.
A cursor has to be created with the `cursor()` method of our connection object conn.

In [6]:
cur = conn.cursor()

Now we can run SQL-Queries with `cur.execute('QUERY')` and then run `cur.fetchall()` to get the data:

In [ ]:
cur.execute("SELECT * FROM eda.king_county_house_sales LIMIT 10")
rows = cur.fetchall()

(datetime.date(2014, 10, 13), 221900.0, 7129300520, 1)
(datetime.date(2014, 12, 9), 538000.0, 6414100192, 2)
(datetime.date(2015, 2, 25), 180000.0, 5631500400, 3)
(datetime.date(2014, 12, 9), 604000.0, 2487200875, 4)
(datetime.date(2015, 2, 18), 510000.0, 1954400510, 5)
(datetime.date(2014, 5, 12), 1230000.0, 7237550310, 6)
(datetime.date(2014, 6, 27), 257500.0, 1321400060, 7)
(datetime.date(2015, 1, 15), 291850.0, 2008000270, 8)
(datetime.date(2015, 4, 15), 229500.0, 2414600126, 9)
(datetime.date(2015, 3, 12), 323000.0, 3793500160, 10)


With `conn.close()` you can close the connection again.

But we want to work with the data. The easiest way is to import the data into pandas dataframes. We can use `pd.read_sql_query` or `pd.read_sql_table` or for convenience `pd.read_sql`.

This function is a convenience wrapper around read_sql_table and read_sql_query (for backward compatibility). It will delegate to the specific function depending on the provided input. A SQL query will be routed to read_sql_query , while a database table name will be routed to read_sql_table . Note that the delegated function might have more specific notes about their functionality not listed here.

In [34]:
# import the data into a pandas dataframe
query_string = """set schema 'eda';

select kchd.* , kchs.date , kchs.price 
from king_county_house_details as kchd
inner join king_county_house_sales kchs on kchd.id = kchs.house_id ;
"""
df = pd.read_sql(query_string, conn)


/var/folders/br/wkt45tkj2bd5stbxtq6czsxm0000gn/T/ipykernel_93610/3775380360.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query_string, conn)


In [35]:
df.to_csv(
    "data/king_county_housing.csv",
    index=False,
    encoding="utf-8"
)

In [37]:
df.shape

(21597, 21)

In [38]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21597 entries, 0 to 21596
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             21597 non-null  int64  
 1   bedrooms       21597 non-null  float64
 2   bathrooms      21597 non-null  float64
 3   sqft_living    21597 non-null  float64
 4   sqft_lot       21597 non-null  float64
 5   floors         21597 non-null  float64
 6   waterfront     19206 non-null  float64
 7   view           21534 non-null  float64
 8   condition      21597 non-null  int64  
 9   grade          21597 non-null  int64  
 10  sqft_above     21597 non-null  float64
 11  sqft_basement  21145 non-null  float64
 12  yr_built       21597 non-null  int64  
 13  yr_renovated   17749 non-null  float64
 14  zipcode        21597 non-null  int64  
 15  lat            21597 non-null  float64
 16  long           21597 non-null  float64
 17  sqft_living15  21597 non-null  float64
 18  sqft_l

In [39]:
df.isnull().sum()

id                  0
bedrooms            0
bathrooms           0
sqft_living         0
sqft_lot            0
floors              0
waterfront       2391
view               63
condition           0
grade               0
sqft_above          0
sqft_basement     452
yr_built            0
yr_renovated     3848
zipcode             0
lat                 0
long                0
sqft_living15       0
sqft_lot15          0
date                0
price               0
dtype: int64

In [41]:
df.isnull().sum()[df.isnull().sum() > 0]

waterfront       2391
view               63
sqft_basement     452
yr_renovated     3848
dtype: int64

In [42]:
df.duplicated().sum()

np.int64(0)

In [43]:
df.describe()

,id,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15,price
count,2.159700e+04,21597.000000,21597.000000,21597.000000,2.159700e+04,21597.000000,19206.000000,21534.000000,21597.000000,21597.000000,21597.000000,21145.000000,21597.000000,17749.000000,21597.000000,21597.000000,21597.000000,21597.000000,21597.000000,2.159700e+04
mean,4.580474e+09,3.373200,2.115826,2080.321850,1.509941e+04,1.494096,0.007602,0.233863,3.409825,7.657915,1788.596842,291.857224,1970.999676,836.650516,98077.951845,47.560093,-122.213983,1986.620318,12758.283512,5.402966e+05
std,2.876736e+09,0.926299,0.768984,918.106125,4.141264e+04,0.539683,0.086858,0.765686,0.650546,1.173200,827.759761,442.490863,29.375234,4000.110554,53.513072,0.138552,0.140724,685.230472,27274.441950,3.673681e+05
min,1.000102e+06,1.000000,0.500000,370.000000,5.200000e+02,1.000000,0.000000,0.000000,1.000000,3.000000,370.000000,0.000000,1900.000000,0.000000,98001.000000,47.155900,-122.519000,399.000000,651.000000,7.800000e+04
25%,2.123049e+09,3.000000,1.750000,1430.000000,5.040000e+03,1.000000,0.000000,0.000000,3.000000,7.000000,1190.000000,0.000000,1951.000000,0.000000,98033.000000,47.471100,-122.328000,1490.000000,5100.000000,3.220000e+05
50%,3.904930e+09,3.000000,2.250000,1910.000000,7.618000e+03,1.500000,0.000000,0.000000,3.000000,7.000000,1560.000000,0.000000,1975.000000,0.000000,98065.000000,47.571800,-122.231000,1840.000000,7620.000000,4.500000e+05
75%,7.308900e+09,4.000000,2.500000,2550.000000,1.068500e+04,2.000000,0.000000,0.000000,4.000000,8.000000,2210.000000,560.000000,1997.000000,0.000000,98118.000000,47.678000,-122.125000,2360.000000,10083.000000,6.450000e+05
max,9.900000e+09,33.000000,8.000000,13540.000000,1.651359e+06,3.500000,1.000000,4.000000,5.000000,13.000000,9410.000000,4820.000000,2015.000000,20150.000000,98199.000000,47.777600,-121.315000,6210.000000,871200.000000,7.700000e+06


In [44]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
id,21597.0,4.580474e+09,2.876736e+09,1.000102e+06,2.123049e+09,3.904930e+09,7.308900e+09,9.900000e+09
bedrooms,21597.0,3.373200e+00,9.262989e-01,1.000000e+00,3.000000e+00,3.000000e+00,4.000000e+00,3.300000e+01
bathrooms,21597.0,2.115826e+00,7.689843e-01,5.000000e-01,1.750000e+00,2.250000e+00,2.500000e+00,8.000000e+00
sqft_living,21597.0,2.080322e+03,9.181061e+02,3.700000e+02,1.430000e+03,1.910000e+03,2.550000e+03,1.354000e+04
sqft_lot,21597.0,1.509941e+04,4.141264e+04,5.200000e+02,5.040000e+03,7.618000e+03,1.068500e+04,1.651359e+06
floors,21597.0,1.494096e+00,5.396828e-01,1.000000e+00,1.000000e+00,1.500000e+00,2.000000e+00,3.500000e+00
waterfront,19206.0,7.601791e-03,8.685849e-02,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00
view,21534.0,2.338627e-01,7.656862e-01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,4.000000e+00
condition,21597.0,3.409825e+00,6.505456e-01,1.000000e+00,3.000000e+00,3.000000e+00,4.000000e+00,5.000000e+00
grade,21597.0,7.657915e+00,1.173200e+00,3.000000e+00,7.000000e+00,7.000000e+00,8.000000e+00,1.300000e+01


In [45]:
df.isnull().sum()[df.isnull().sum() > 0]

waterfront       2391
view               63
sqft_basement     452
yr_renovated     3848
dtype: int64

In [46]:
df[['waterfront', 'view', 'sqft_basement', 'yr_renovated']].head(20)

,waterfront,view,sqft_basement,yr_renovated
0,NaN,0.0,0.0,0.0
1,0.0,0.0,400.0,19910.0
2,0.0,0.0,0.0,NaN
3,0.0,0.0,910.0,0.0
4,0.0,0.0,0.0,0.0
5,0.0,0.0,1530.0,0.0
6,0.0,0.0,NaN,0.0
7,0.0,NaN,0.0,0.0
8,0.0,0.0,730.0,0.0
9,0.0,0.0,0.0,0.0


In [47]:
df[['waterfront', 'view', 'sqft_basement', 'yr_renovated']].sample(20, random_state=42)

,waterfront,view,sqft_basement,yr_renovated
3686,0.0,0.0,250.0,0.0
10247,0.0,0.0,650.0,0.0
4037,0.0,0.0,620.0,0.0
3437,0.0,1.0,510.0,NaN
19291,0.0,0.0,500.0,0.0
13858,0.0,0.0,0.0,0.0
3503,0.0,0.0,0.0,0.0
19247,0.0,1.0,2220.0,0.0
6433,0.0,0.0,250.0,0.0
5843,0.0,0.0,0.0,0.0


In [48]:
for col in ['waterfront', 'view', 'sqft_basement', 'yr_renovated']:
    print(f"\n{col}")
    print(df[col].value_counts(dropna=False).sort_index())


waterfront
waterfront
0.0    19060
1.0      146
NaN     2391
Name: count, dtype: int64

view
view
0.0    19422
1.0      330
2.0      957
3.0      508
4.0      317
NaN       63
Name: count, dtype: int64

sqft_basement
sqft_basement
0.0       12827
10.0          2
20.0          1
40.0          4
50.0         11
          ...  
3480.0        1
3500.0        1
4130.0        1
4820.0        1
NaN         452
Name: count, Length: 304, dtype: int64

yr_renovated
yr_renovated
0.0        17005
19340.0        1
19400.0        2
19440.0        1
19450.0        3
           ...  
20120.0        8
20130.0       31
20140.0       73
20150.0       14
NaN         3848
Name: count, Length: 71, dtype: int64


In [49]:
df['waterfront'] = df['waterfront'].fillna(0)
df['view'] = df['view'].fillna(0)
df['sqft_basement'] = df['sqft_basement'].fillna(0)
df['yr_renovated'] = df['yr_renovated'].fillna(0)

In [50]:
df.isnull().sum()

id               0
bedrooms         0
bathrooms        0
sqft_living      0
sqft_lot         0
floors           0
waterfront       0
view             0
condition        0
grade            0
sqft_above       0
sqft_basement    0
yr_built         0
yr_renovated     0
zipcode          0
lat              0
long             0
sqft_living15    0
sqft_lot15       0
date             0
price            0
dtype: int64

In [51]:
df['date'] = pd.to_datetime(df['date'])

In [52]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21597 entries, 0 to 21596
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   id             21597 non-null  int64         
 1   bedrooms       21597 non-null  float64       
 2   bathrooms      21597 non-null  float64       
 3   sqft_living    21597 non-null  float64       
 4   sqft_lot       21597 non-null  float64       
 5   floors         21597 non-null  float64       
 6   waterfront     21597 non-null  float64       
 7   view           21597 non-null  float64       
 8   condition      21597 non-null  int64         
 9   grade          21597 non-null  int64         
 10  sqft_above     21597 non-null  float64       
 11  sqft_basement  21597 non-null  float64       
 12  yr_built       21597 non-null  int64         
 13  yr_renovated   21597 non-null  float64       
 14  zipcode        21597 non-null  int64         
 15  lat            2159

In [53]:
df['zipcode'] = df['zipcode'].astype('category')

In [54]:
df[df['yr_renovated'] != 0]['yr_renovated'].sort_values().unique()

array([19340., 19400., 19440., 19450., 19460., 19480., 19500., 19510.,
       19530., 19540., 19550., 19560., 19570., 19580., 19590., 19600.,
       19620., 19630., 19640., 19650., 19670., 19680., 19690., 19700.,
       19710., 19720., 19730., 19740., 19750., 19760., 19770., 19780.,
       19790., 19800., 19810., 19820., 19830., 19840., 19850., 19860.,
       19870., 19880., 19890., 19900., 19910., 19920., 19930., 19940.,
       19950., 19960., 19970., 19980., 19990., 20000., 20010., 20020.,
       20030., 20040., 20050., 20060., 20070., 20080., 20090., 20100.,
       20110., 20120., 20130., 20140., 20150.])

In [55]:
df.loc[df['yr_renovated'] != 0, 'yr_renovated'] = (
    df.loc[df['yr_renovated'] != 0, 'yr_renovated'] / 10
).astype(int)

In [56]:
df['yr_renovated'].describe()

count    21597.000000
mean        68.758207
std        364.037499
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max       2015.000000
Name: yr_renovated, dtype: float64

In [57]:
df['yr_renovated'] = df['yr_renovated'].astype(int)

In [58]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21597 entries, 0 to 21596
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   id             21597 non-null  int64         
 1   bedrooms       21597 non-null  float64       
 2   bathrooms      21597 non-null  float64       
 3   sqft_living    21597 non-null  float64       
 4   sqft_lot       21597 non-null  float64       
 5   floors         21597 non-null  float64       
 6   waterfront     21597 non-null  float64       
 7   view           21597 non-null  float64       
 8   condition      21597 non-null  int64         
 9   grade          21597 non-null  int64         
 10  sqft_above     21597 non-null  float64       
 11  sqft_basement  21597 non-null  float64       
 12  yr_built       21597 non-null  int64         
 13  yr_renovated   21597 non-null  int64         
 14  zipcode        21597 non-null  category      
 15  lat            2159

In [59]:
df['waterfront'] = df['waterfront'].astype(int)
df['view'] = df['view'].astype(int)

In [60]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21597 entries, 0 to 21596
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   id             21597 non-null  int64         
 1   bedrooms       21597 non-null  float64       
 2   bathrooms      21597 non-null  float64       
 3   sqft_living    21597 non-null  float64       
 4   sqft_lot       21597 non-null  float64       
 5   floors         21597 non-null  float64       
 6   waterfront     21597 non-null  int64         
 7   view           21597 non-null  int64         
 8   condition      21597 non-null  int64         
 9   grade          21597 non-null  int64         
 10  sqft_above     21597 non-null  float64       
 11  sqft_basement  21597 non-null  float64       
 12  yr_built       21597 non-null  int64         
 13  yr_renovated   21597 non-null  int64         
 14  zipcode        21597 non-null  category      
 15  lat            2159

In [61]:
df.to_csv(
    "data/king_county_housing_clean.csv",
    index=False,
    encoding="utf-8"
)

In [ ]:
# close the connection
conn.close()

In [ ]:
df_psycopg.head()

In [ ]:
# export the data to a csv-file
df_psycopg.to_csv("data/eda.csv", index=False)

### Connecting and retrieving data via SQLAlchemy

`sqlalchemy` works similarly. Here you have to create an engine with the database string (a link that includes every information we entered in the conn object)

In [ ]:
from sqlalchemy import create_engine

# read the database string from the .env
load_dotenv()

DB_STRING = os.getenv("DB_STRING")

if DB_STRING is None:
    raise ValueError("DB_STRING is not set in the environment.")

db = create_engine(DB_STRING)

And then you can import that engine with a query into a pandas dataframe.

In [ ]:
# import the data to a pandas dataframe
query_string = "SELECT * FROM eda.king_county_house_sales"
df_sqlalchemy = pd.read_sql(query_string, db)

In [ ]:
df_sqlalchemy.head()

Because we don't want to run the queries over and over again we can export the data into a .csv file in order to use it in other notebooks as well. 

In [ ]:
# export the data to a csv-file
df_sqlalchemy.to_csv("data/eda.csv", index=False)

In [ ]:
# import the data from a csv-file
df_import = pd.read_csv("data/eda.csv")

The dataset was checked for missing values, duplicates, data types, and unreasonable values. No major data quality issues affecting the analysis were found.